# 03 — Train replacement heads (Skew-t + IQN), all 7 assets

Trains head-only on the cached embeddings from notebook 02 (fast — no backbone
forward). For each asset: train Skew-Student-t (NLL) and IQN (sampled-τ pinball),
then evaluate **arbitrary-τ tail calibration** the native decile head can't give:
PIT/KS before vs after, per-τ reliability, and VaR/ES at 1/5/95/99%.

Skew-t is the stable primary (analytic CDF + ES); IQN is the novel headline.

## 1. Install + clone (then RESTART KERNEL). Needs the emb caches from nb 02
(either rerun nb 02 first in this session, or add the `emb_caches.zip` output).

In [ ]:
!pip install "git+https://github.com/google-research/timesfm.git#egg=timesfm[torch]" -q
!pip install "tsfm_cal @ git+https://github.com/YOURNAME/tsfm-thesis.git" -q
# >>> Restart kernel, then continue. <<<

## 2. Imports

In [ ]:
import numpy as np, torch
from tsfm_cal import config, backbone, heads, finetune, eval as ev, io_utils
RISK = config.RISK_QUANTILES
print("risk grid:", RISK)

## 3. Train + evaluate one head on one asset

In [ ]:
def run_head(key, kind, epochs=300, lr=2e-3):
    label = config.asset_display(key)
    cache = backbone.load_cache(key)
    Head = heads.SkewStudentTHead if kind == "skewt" else heads.IQNHead
    head = Head()
    print(f"[{label}] training {kind} on n={len(cache['emb'])} ...")
    head, hist = finetune.train_head(head, cache, kind, epochs=epochs, lr=lr,
                                     batch_size=512, patience=40, verbose=True)

    # PIT after head replacement + KS
    pit = heads.head_pit(head, cache)
    ks, p = ev.ks_uniform(pit)

    # per-tau reliability at the risk grid
    Q = heads.head_quantiles(head, cache, RISK)            # (N, len(RISK)) denormalized
    reliab = ev.quantile_reliability(cache["target_raw"], Q, RISK)

    # tail VaR/ES backtests (loss tail 1% & 5%, upper tail 95% & 99%)
    realized = cache["target_raw"]
    tails = {}
    for a, side in [(0.01,"left"),(0.05,"left"),(0.95,"right"),(0.99,"right")]:
        var = heads.head_var(head, cache, a)
        es  = heads.head_es(head, cache, a, side=side)
        tails[f"{int(a*100)}%_{side}"] = ev.backtest_var_es(realized, var, es, a, side)

    finetune.save_run(key, head, {**hist, "ks_after": ks, "ks_p": p}, cache, pit)
    print(f"  {kind} KS_after={ks:.4f} (p={p:.1e})")
    return {"key": key, "kind": kind, "ks": ks, "reliab": reliab, "tails": tails}

## 4. Run skew-t (primary) then IQN (headline) for all assets

In [ ]:
results = []
for kind in ["skewt", "iqn"]:
    for key in config.ASSET_KEYS:
        try:
            results.append(run_head(key, kind))
        except FileNotFoundError:
            print(f"! no cache for {key} — run notebook 02 first")
        except Exception as e:
            print(f"! {key}/{kind} failed: {e}")

## 5. Before/after KS table — head replacement vs zero-shot

`pit_before` = the zero-shot PIT from your Phase-1 run. Drop a zeroshot
`pit/<ASSET>.npz` into each `outputs/finetune/<KEY>/pit_before.npz`, or compare
against the Phase-1 `pit_summary.csv` KS column directly.

In [ ]:
import pandas as pd
rows = [{"asset": config.asset_display(r["key"]), "head": r["kind"],
         "KS_after": round(r["ks"], 4)} for r in results]
print(pd.DataFrame(rows).to_string(index=False))

## 6. Tail VaR/ES summary at 1% (Kupiec / Christoffersen / ES±CI / FZ)

In [ ]:
rows = []
for r in results:
    t = r["tails"]["1%_left"]
    rows.append({"asset": config.asset_display(r["key"]), "head": r["kind"],
                 "exc_rate": round(t["rate"],4), "kupiec_p": round(t["p_uc"],3),
                 "cc_p": round(t["p_cc"],3) if t["p_cc"]==t["p_cc"] else None,
                 "ES": round(t["es_realized"],4),
                 "ES_CI": (round(t["es_ci_lo"],4), round(t["es_ci_hi"],4)),
                 "FZ": round(t["fz_loss"],4)})
print(pd.DataFrame(rows).to_string(index=False))

## 7. Per-τ reliability for SPY (skew-t) — empirical vs nominal coverage

In [ ]:
spy = next(r for r in results if r["key"]=="SPY" and r["kind"]=="skewt")
print(pd.DataFrame(spy["reliab"]).to_string(index=False))

## 8. Zip head artifacts for download

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/head_runs", "zip", io_utils.base_dir()/"finetune")
print("Download:", zip_path)